In [ ]:
%pip install transformers datasets scikit-learn torch accelerate seaborn matplotlib

In [1]:
import json
import torch
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

print(f"torch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")

c:\Users\Júlia Calixto\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.12.1+cpu
CUDA disponível: False


## Definições importantes


In [ ]:
_here = Path.cwd()
PROJECT_ROOT = _here.parent if _here.name == "scripts" else _here

CSV_PATH = PROJECT_ROOT / "data/seed_relatos_fala_gavea_1k.csv"
OUTPUT_DIR = PROJECT_ROOT / "models/topic_classifier"

MODEL_NAME = "distilbert/distilbert-base-multilingual-cased"

MAX_LENGTH = 64
BATCH_SIZE = 16
EPOCHS = 5

## Carregando e preparando dados

In [3]:
df = pd.read_csv(CSV_PATH).dropna(subset=["texto_relato", "topico"])

le = LabelEncoder()
df["label"] = le.fit_transform(df["topico"])
num_labels = len(le.classes_)

print(f"Total de exemplos: {len(df)}")
print(f"\nDistribuição por tópico:")
print(df["topico"].value_counts().to_string())

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
label_map = {int(i): name for i, name in enumerate(le.classes_)}
(OUTPUT_DIR / "label_map.json").write_text(json.dumps(label_map, ensure_ascii=False, indent=2))

train_df, val_df = train_test_split(
    df[["texto_relato", "label"]],
    test_size=0.2,
    stratify=df["label"],   # garante proporção igual em treino e validação
    random_state=42,
)

print(f"\nSplit: {len(train_df)} treino | {len(val_df)} validação (80/20 estratificado)")

Total de exemplos: 5000

Distribuição por tópico:
topico
Conflito social           1265
Transito e mobilidade      943
Lixo e conservacao         831
Vandalismo                 683
Iluminacao publica         665
Espaco publico             309
Seguranca e circulacao     304

Split: 4000 treino | 1000 validação (80/20 estratificado)


## Tokenização

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["texto_relato"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True))

train_ds = train_ds.map(tokenize, batched=True).remove_columns(["texto_relato"])
val_ds   = val_ds.map(tokenize, batched=True).remove_columns(["texto_relato"])



Map: 100%|██████████| 1000/1000 [00:00<00:00, 19195.28 examples/s]


### Métricas

In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    report = classification_report(labels, preds, output_dict=True, zero_division=0)
    return {
        "f1_macro":  report["macro avg"]["f1-score"],
        "accuracy":  report["accuracy"],
    }

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=label_map,
    label2id={v: k for k, v in label_map.items()},
)

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,   # era 'tokenizer=' no transformers < 5.0
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()
trainer.save_model(str(OUTPUT_DIR / "best"))
print("\nTreinamento concluído. Modelo salvo em:", OUTPUT_DIR / "best")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4033.88it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
c:\Users\Júlia Calixto\AppDat

Epoch,Training Loss,Validation Loss


## Avaliação de resultados

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

preds_output = trainer.predict(val_ds)
preds = np.argmax(preds_output.predictions, axis=-1)
true_labels = val_df["label"].values

print("=" * 55)
print("RELATÓRIO DE CLASSIFICAÇÃO — conjunto de validação")
print("=" * 55)
print(classification_report(true_labels, preds, target_names=le.classes_, zero_division=0))

### Matriz de confusão

In [ ]:
cm = confusion_matrix(true_labels, preds)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=le.classes_, yticklabels=le.classes_, ax=ax,
)
ax.set_ylabel("Real", fontsize=12)
ax.set_xlabel("Previsto", fontsize=12)
ax.set_title("Matriz de Confusão — validação", fontsize=14)
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### Exemplos de previsão: acertos e erros

In [ ]:
probs_all = torch.tensor(preds_output.predictions).softmax(dim=-1).numpy()

def mostrar_exemplos(indices, titulo, n=5):
    print(f"\n{'='*60}")
    print(f"  {titulo}")
    print(f"{'='*60}")
    for i, idx in enumerate(indices[:n]):
        texto = val_df["texto_relato"].iloc[idx]
        real = le.classes_[true_labels[idx]]
        previsto = le.classes_[preds[idx]]
        confianca = probs_all[idx][preds[idx]]
        print(f"\n[{i+1}] {texto[:100]}{'...' if len(texto) > 100 else ''}")
        print(f"    Real:     {real}")
        print(f"    Previsto: {previsto}  ({confianca:.1%} de confiança)")

acertos = np.where(preds == true_labels)[0]
erros   = np.where(preds != true_labels)[0]

mostrar_exemplos(acertos, "ACERTOS")
mostrar_exemplos(erros,   "ERROS")

print(f"\n\nResumo: {len(acertos)}/{len(true_labels)} corretos ({len(acertos)/len(true_labels):.1%} accuracy no val)")

## Helper para inferência dps

In [ ]:
# (imports já feitos no topo — célula mantida para referência do helper)

In [ ]:
class TopicClassifier:
    def __init__(self, model_dir: Path | str | None = None):
        if model_dir is None:
            model_dir = PROJECT_ROOT / "models/topic_classifier/best"
        self.tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
        self.model = AutoModelForSequenceClassification.from_pretrained(str(model_dir))
        self.model.eval()
        label_map_path = Path(model_dir).parent / "label_map.json"
        self.label_map: dict[int, str] = {
            int(k): v for k, v in json.loads(label_map_path.read_text()).items()
        }

    def predict(self, texto: str) -> tuple[str, float]:
        inputs = self.tokenizer(texto, return_tensors="pt", truncation=True, max_length=128)
        with torch.no_grad():
            logits = self.model(**inputs).logits
        probs = logits.softmax(dim=-1)[0]
        label_id = probs.argmax().item()
        return self.label_map[label_id], round(probs[label_id].item(), 4)

### Teste com textos novos

In [ ]:
classifier = TopicClassifier()

exemplos = [
    "Tem muito lixo acumulado na calçada da rua das Acácias há dias, ninguém recolhe.",
    "Semáforo da esquina da Marquês de São Vicente com a Lagoa está quebrado há semanas.",
    "Assaltaram dois pedestres ontem à noite perto do Parque da Catacumba.",
    "A iluminação pública da rua Marquês de São Vicente está completamente apagada.",
    "Picharam toda a parede do colégio estadual com grafite obsceno.",
    "Briga generalizada entre grupos rivais na pracinha da Gávea domingo à noite.",
    "Calçada da praça completamente destruída, impossível caminhar com carrinho de bebê.",
]

print("Previsões para textos novos:\n")
for texto in exemplos:
    topico, confianca = classifier.predict(texto)
    barra = "█" * int(confianca * 20)
    print(f"  {texto[:75]}...")
    print(f"  → {topico:<28} {barra} {confianca:.1%}\n")